In [1]:
import os
import cv2 as cv
import numpy as np
from skimage import measure
from skimage.segmentation import clear_border
from sklearn.cluster import KMeans

In [2]:
def generate_lung_mask(ct_img):
    """
    Generates a 512x512 lung mask using morphological operations
    as described in Fig. 7 (A–H)

    Input:
        ct_img : grayscale CT image (512x512)
    Output:
        lung_mask : binary lung mask (512x512)
        masked_img : lung-only CT image
    """

    # --------------------------------------------------
    # A: Image Negation
    # --------------------------------------------------
    neg_img = cv.bitwise_not(ct_img)

    # --------------------------------------------------
    # B: K-means Thresholding
    # --------------------------------------------------
    flat = neg_img.reshape(-1, 1).astype(np.float32)

    kmeans = KMeans(
        n_clusters=2,
        n_init=10,
        random_state=42
    ).fit(flat)

    centers = np.sort(kmeans.cluster_centers_.flatten())
    threshold = np.mean(centers)

    _, binary = cv.threshold(
        neg_img,
        threshold,
        255,
        cv.THRESH_BINARY
    )

    # --------------------------------------------------
    # C: Clear Border Objects
    # --------------------------------------------------
    cleared = clear_border(binary)

    # --------------------------------------------------
    # D: Connected Component Labeling
    # --------------------------------------------------
    labels = measure.label(cleared)
    regions = measure.regionprops(labels)

    # --------------------------------------------------
    # E: Keep Two Largest Regions (Lungs)
    # --------------------------------------------------
    regions = sorted(regions, key=lambda r: r.area, reverse=True)

    lung_mask = np.zeros_like(labels, dtype=np.uint8)

    for r in regions[:2]:
        lung_mask[labels == r.label] = 255

    # --------------------------------------------------
    # F: Remove Small Structures (Vessels / Nodules)
    # --------------------------------------------------
    kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (5, 5))
    lung_mask = cv.morphologyEx(
        lung_mask,
        cv.MORPH_OPEN,
        kernel
    )

    # --------------------------------------------------
    # G: Mask Refinement (Closing + Hole Filling)
    # --------------------------------------------------
    lung_mask = cv.morphologyEx(
        lung_mask,
        cv.MORPH_CLOSE,
        kernel
    )

    contours, _ = cv.findContours(
        lung_mask,
        cv.RETR_EXTERNAL,
        cv.CHAIN_APPROX_SIMPLE
    )
    cv.drawContours(
        lung_mask,
        contours,
        -1,
        255,
        thickness=cv.FILLED
    )

    # --------------------------------------------------
    # H: Superimpose Mask on Negated Image
    # --------------------------------------------------
    masked_img = cv.bitwise_and(
        neg_img,
        neg_img,
        mask=lung_mask
    )

    return  masked_img

In [17]:

img_path = r"D:\ResearchPaper\resized_512_padded\N\N_N_ds2_17 - Copy - Copy.png"
ct_image = cv.imread(img_path, cv.IMREAD_GRAYSCALE) 
masked_img = generate_lung_mask(ct_image)

# Display the result
cv.imshow('img', masked_img)
cv.waitKey(0)
cv.destroyAllWindows()